In [ ]:
# install the required libraries if they are not available in your environment
# pandas -> data manipulation and analysis
# scikit-learn -> machine learning library

# !pip install pandas scikit-learn

In [ ]:
# project workflow
# step 0 - understand the business problem
# the objective is to build a machine learning model capable of predicting
# a customer's credit score based on historical information.

# step 1 - import the dataset

# import the pandas library.
# "import ... as ..." creates an alias, allowing us to use "pd"
# instead of typing "pandas" every time.

import pandas as pd

# read the csv file and store it inside a dataframe.
# a dataframe is a table composed of rows and columns.

tabela = pd.read_csv("clientes.csv")

# display the dataset for an initial inspection

display(tabela)

# target classes
# good -> good credit score
# standard -> average credit score
# poor -> low credit score

In [ ]:
# step 2 - prepare the dataset for machine learning

# inspect the dataset structure
# info() shows:
# - number of rows
# - number of non-null values
# - data types
# - memory usage

display(tabela.info())

# common data types
# int -> integer numbers
# float -> decimal numbers
# object -> text values

# machine learning algorithms cannot work directly with text.
# therefore, categorical values must be converted into numbers.

from sklearn.preprocessing import LabelEncoder

# LabelEncoder assigns a numeric value to each unique category.
# example:
# engineer -> 0
# doctor -> 1
# teacher -> 2

# profession

codificador_profissao = LabelEncoder()

# fit_transform()
# fit -> learns all existing categories
# transform -> converts categories into numbers

tabela["profissao"] = codificador_profissao.fit_transform(
    tabela["profissao"]
)

# credit mix

codificador_credito = LabelEncoder()

tabela["mix_credito"] = codificador_credito.fit_transform(
    tabela["mix_credito"]
)

# payment behavior

codificador_pagamento = LabelEncoder()

tabela["comportamento_pagamento"] = codificador_pagamento.fit_transform(
    tabela["comportamento_pagamento"]
)

# verify that categorical columns were converted into numeric values

display(tabela.info())

In [ ]:
# step 3 - separate features from the target

# y represents the target variable.
# this is the information the model must predict.

y = tabela["score_credito"]

# x contains the input features used by the model.
# score_credito is removed because it is the expected answer.
# id_cliente is removed because it does not influence credit score.

x = tabela.drop(columns=["score_credito", "id_cliente"])

# split the dataset into training and testing sets

from sklearn.model_selection import train_test_split

# training data -> used to teach the model
# testing data -> used to evaluate the model with unseen data

# test_size=0.3 means:
# 70% training
# 30% testing

x_treino, x_teste, y_treino, y_teste = train_test_split(x, y, test_size=0.3)

In [ ]:
# step 4 - train machine learning models

# this project compares two classification algorithms.

# Random Forest
# builds multiple decision trees and combines their predictions.
# it is usually more accurate and more robust.

# K-Nearest Neighbors (KNN)
# classifies a sample according to the most similar neighboring samples.
# similarity is calculated using distance.

# importing machine learning algorithms

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# create the models

modelo_arvoredecisao = RandomForestClassifier()
modelo_knn = KNeighborsClassifier()

# fit() trains the models using the training dataset.
# during this stage, the algorithms learn patterns from historical data.

modelo_arvoredecisao.fit(x_treino, y_treino)

modelo_knn.fit(x_treino, y_treino)

# quick comparison

# Random Forest
# + higher accuracy
# + handles complex relationships
# + less sensitive to noisy data

# KNN
# + simple algorithm
# + effective on smaller datasets
# - slower during prediction
# - sensitive to feature scaling

In [ ]:
# step 5 - compare the trained models

# predict() generates predictions using data the model has never seen.

previsao_arvoredecisao = modelo_arvoredecisao.predict(x_teste)

previsao_knn = modelo_knn.predict(x_teste)

# accuracy measures the percentage of correct predictions.

from sklearn.metrics import accuracy_score

display(accuracy_score(y_teste, previsao_arvoredecisao))

display(accuracy_score(y_teste, previsao_knn))

# the model with the highest accuracy is selected for production.

In [ ]:
# step 6 - predict the credit score of new customers

# the Random Forest model achieved the best performance
# and will therefore be used for new predictions.

# load new customer data

tabela_novos_clientes = pd.read_csv("novos_clientes.csv")

# important:
# new data must receive the same preprocessing
# used during training.

# profession

tabela_novos_clientes["profissao"] = codificador_profissao.transform(
    tabela_novos_clientes["profissao"]
)

# credit mix

tabela_novos_clientes["mix_credito"] = codificador_credito.transform(
    tabela_novos_clientes["mix_credito"]
)

# payment behavior

tabela_novos_clientes["comportamento_pagamento"] = codificador_pagamento.transform(
    tabela_novos_clientes["comportamento_pagamento"]
)

display(tabela_novos_clientes)

# predict the credit score for each new customer

nova_previsao = modelo_arvoredecisao.predict(
    tabela_novos_clientes
)

display(nova_previsao)